# surfaces

> Cards for grouping content, badges for short labels, and avatars for people and teams

In [ ]:
#| default_exp surfaces

`Card` wraps content in a bordered section styled with the shared colour, spacing, and radius tokens. `Badge` renders an inline text label for a category, count, or status. `Avatar` renders a circular image or initials for a person or team.

In [ ]:
from answerui.registry import hypelt
from answerui.layout import Stack, Cluster
from answerui.demo import example
from fasthtml.common import Section, H3, P, Span, Img

In [ ]:
from fasthtml.common import Article, Header, Footer, Small, to_xml
from answerui.layout import Grid
from answerui.controls import Btn
from answerui.demo import demo
from urllib.parse import quote

## Card

Add the `card` attribute to any element for a `--color-surface` background, a 1px `--color-border`, `--space` padding, `--radius` corners, and a `--raised` shadow. `[card]` uses no bevels so cards do not look clickable. `Card(...)` returns a `<section card>` element and uses no JavaScript.

`[card]` sets no child layout. Pass `stack=True` for a vertical stack, or nest another layout inside the card.

In [ ]:
@hypelt(sel='[card]')
def Card(*c, **attrs):
    "A bordered section with no prescribed layout"
    return Section(*c, card=True, **attrs)

Inner padding comes from `[card]`. Add `stack=True` for spacing between children, and use a surrounding layout such as `Grid` or `Stack` to space cards apart.

In [ ]:
@example(Card)
def _card_example():
    return Card(H3('Shared workspace'), P('Keep your projects and notes together in one place.'), stack=True)

In [ ]:
Card.demo()

```html
<section card stack>
  <h3>Shared workspace</h3>
  <p>Keep your projects and notes together in one place.</p>
</section>

```

### Composition

You can use any HTML structure inside a card without dedicated header, body, or footer constructors. Below, `Grid` arranges two cards side by side when the container is wide enough. The second card is an `Article` with `card=True` and `stack=True` directly, using `stack=True` on its `Header` and `cluster=True` on its `Footer`.

In [ ]:
demo(Grid(
    _card_example(),
    Article(
        Header(H3('Field notes'), Small('Updated today'), stack=True, cls='gap-1'),
        P('Observations and ideas from the team, ready for the next meeting.'),
        Footer(Btn('Open project'), Btn('Archive', variant='secondary'), cluster=True),
        card=True, stack=True)))

```html
<div grid>
  <section card stack>
    <h3>Shared workspace</h3>
    <p>Keep your projects and notes together in one place.</p>
  </section>
  <article card stack>
    <header stack style="--gap: var(--space-3)">
      <h3>Field notes</h3>
<small>Updated today</small>    </header>
    <p>Observations and ideas from the team, ready for the next meeting.</p>
    <footer cluster>
<button type="button" btn btn-variant="primary">Open project</button><button type="button" btn btn-variant="secondary">Archive</button>    </footer>
  </article>
</div>

```

### Customisation

Overriding `--radius` or `--space` changes card corners and padding across a region. To restyle all cards without affecting other components, add a rule such as `[card] { padding: var(--space1); }` in `@layer app`. For a single card, pass utility classes such as `cls='gap-2 p-2'` (or inline `style` for custom values):

In [ ]:
to_xml(Card(P('A compact note'), P('Another line'), stack=True, cls='gap-2 p-2'))

'<section card stack style="--gap: var(--space-2); padding: var(--space-2)">\n  <p>A compact note</p>\n  <p>Another line</p>\n</section>\n'

You can combine `card=True` with any layout constructor, such as `Grid(..., card=True)`, without adding a wrapper `div`. Plain `article`, `section`, and `div` elements without `card` have no card border or background.

## Badge

`Badge` renders an inline `<span>` with a soft `--v-3` background and `--v2` text rather than the solid, raised fill used on buttons. Pass `ui_tone` (such as `'success'`, `'warning'`, `'danger'`, or `'info'`) to change its hue, or `variant='secondary'` for neutral `--color-surface` and `--color-text` styling with a 1px inset `--color-border`.

In [ ]:
@hypelt(sel='[badge]')
def Badge(*c, variant=None, **attrs):
    "A small text label, optionally coloured with `ui_tone`"
    return Span(*c, badge=True, badge_variant=variant, **attrs)

Badges use `em` padding (`.125em .5em`) at `--text-1` size so padding scales with the text. Put the meaning in the badge text and use colour only to reinforce it. `[badge]` is display-only, with no interaction or ARIA live-region role.

In [ ]:
@example(Badge)
def _badge_example():
    return Cluster(Badge('New'), Badge('Draft', variant='secondary'), Badge('Active', ui_tone='success'),
        Badge('Pending', ui_tone='warning'), Badge('Failed', ui_tone='danger'), Badge('Preview', ui_tone='info'),
        cls='gap-2')

In [ ]:
Badge.demo()

```html
<div cluster style="--gap: var(--space-2);">
<span badge>New</span><span badge badge-variant="secondary">Draft</span><span badge ui-tone="success">Active</span><span badge ui-tone="warning">Pending</span><span badge ui-tone="danger">Failed</span><span badge ui-tone="info">Preview</span></div>

```

Badges have no outer margin. Below, `Cluster` places a badge beside the heading while `Card(..., stack=True)` spaces the rows apart. Pass counts as ordinary text children rather than CSS-generated content:

In [ ]:
demo(Card(
    Cluster(H3('Field notes'), Badge('Shared'), cls='gap-2'),
    P('Observations and ideas from the team.'),
    Cluster(Span('Unread notes'), Badge(3, variant='secondary'), cls='gap-2'), stack=True))

```html
<section card stack>
  <div cluster style="--gap: var(--space-2);">
    <h3>Field notes</h3>
<span badge>Shared</span>  </div>
  <p>Observations and ideas from the team.</p>
  <div cluster style="--gap: var(--space-2);">
<span>Unread notes</span><span badge badge-variant="secondary">3</span>  </div>
</section>

```

## Avatar

`Avatar` takes a `name` and an optional image URL in `src`. With `src`, it renders an `<img>` with `alt=name`. Without `src`, it renders a `<span role="img">` showing the first and last initials (or `?` if `name` is empty) with `aria-label` set to `name` (or `'Unknown'`). Both forms share the same circular frame controlled by `--avatar-size` (default `2.5rem`), with initials sized to 40% of `--avatar-size`.

In [ ]:
@hypelt(sel='[avatar]')
def Avatar(
    name,     # Full name, used as the accessible label and to derive initials
    src=None, # Image URL; omit to show initials
    **attrs,  # Attributes for the image or span
):
    "A circular image or initials identifying a person or team"
    if src: return Img(src=src, alt=name, avatar=True, **attrs)
    parts = name.split()
    initials = parts[0][0] if parts else '?'
    if len(parts) > 1: initials += parts[-1][0]
    return Span(initials.upper(), role='img', aria_label=name or 'Unknown', avatar=True, **attrs)

In [ ]:
@example(Avatar)
def _avatar_example():
    return Cluster(Avatar('Ada Lovelace'), Avatar('Grace Hopper'), Avatar('Prince'),
        Avatar('Alan Turing', style='--avatar-size: 3.5rem'))

In [ ]:
Avatar.demo()

```html
<div cluster>
<span role="img" aria-label="Ada Lovelace" avatar>AL</span><span role="img" aria-label="Grace Hopper" avatar>GH</span><span role="img" aria-label="Prince" avatar>P</span><span role="img" aria-label="Alan Turing" avatar style="--avatar-size: 3.5rem">AT</span></div>

```

In [ ]:
def _picture(name): return f'https://api.dicebear.com/10.x/miniavs/svg?seed={quote(name)}&backgroundColor=ffe3ea'
demo(Cluster(Avatar('Ada Lovelace', src=_picture('Ada Lovelace'), aria_hidden='true'),
    Stack(Span('Ada Lovelace'), Small('Workspace owner'), cls='gap-1')))

```html
<div cluster>
<img src="https://api.dicebear.com/10.x/miniavs/svg?seed=Ada%20Lovelace&amp;backgroundColor=ffe3ea" alt="Ada Lovelace" avatar aria-hidden="true">  <div stack style="--gap: var(--space-3);">
<span>Ada Lovelace</span><small>Workspace owner</small>  </div>
</div>

```

To make an avatar clickable, place it as the only child of a `Btn`. The `[btn]:has(> [avatar]:only-child)` rule sets `1px` padding, `border-radius: var(--radius-full)`, `--face: var(--color-surface)`, and `--color-border` on the button without overriding the avatar's colours. When the button is disabled, the avatar has `opacity: .5`.

In [ ]:
demo(Cluster(Btn(Avatar('Ada Lovelace')), Btn(Avatar('Grace Hopper', src=_picture('Grace Hopper'))),
    Btn(Avatar('Alan Turing', src=_picture('Alan Turing')), disabled=True)))

```html
<div cluster>
<button type="button" btn btn-variant="primary"><span role="img" aria-label="Ada Lovelace" avatar>AL</span></button><button type="button" btn btn-variant="primary"><img src="https://api.dicebear.com/10.x/miniavs/svg?seed=Grace%20Hopper&amp;backgroundColor=ffe3ea" alt="Grace Hopper" avatar></button><button type="button" btn btn-variant="primary" disabled><img src="https://api.dicebear.com/10.x/miniavs/svg?seed=Alan%20Turing&amp;backgroundColor=ffe3ea" alt="Alan Turing" avatar></button></div>

```